# 02 — Plasma OES Classification

**LIBS Benchmark** — SVM + CNN species classification + SHAP wavelength importance.

This notebook demonstrates:
1. Loading and preprocessing a small LIBS Benchmark subset
2. Training an SVM classifier with 5-fold cross-validation
3. Plotting the confusion matrix
4. Training a lightweight CNN on PCA features
5. Computing SHAP wavelength importance via LinearSVC + PCA back-projection

In [1]:
import os
import sys
import warnings
from pathlib import Path

nb_dir = Path.cwd()
if nb_dir.name == "notebooks":
    os.chdir(nb_dir.parent)
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

print("Working directory:", Path.cwd())

Working directory: C:\Users\PC\libs-spectral-analysis


## 1 · Load and preprocess LIBS Benchmark

In [2]:
from src.data_loader import load_libs_benchmark
from src.preprocessing import Preprocessor
from sklearn.preprocessing import StandardScaler

# Load small subset (max 2 per sample group → ~192 spectra total)
X, y, wl = load_libs_benchmark("data/libs_benchmark/", max_spectra_per_class=2)
print(f"Dataset : {X.shape[0]} spectra × {X.shape[1]} channels")

# Lightweight preprocessing (SNV + SavGol, skip ALS for speed in CV)
pp = Preprocessor(baseline="none", normalize="snv", denoise="savgol", cosmic_ray=False)
X_pp = pp.fit_transform(X)
X_sc = StandardScaler().fit_transform(X_pp)
print(f"Preprocessed : {X_sc.shape}")

Dataset : 192 spectra × 40002 channels
Preprocessed : (192, 40002)


## 2 · PCA dimensionality reduction

In [3]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_sc)
ev_ratio = pca.explained_variance_ratio_.sum()
print(f"PCA(50) : {X_pca.shape}, explained variance = {ev_ratio:.1%}")

PCA(50) : (192, 50), explained variance = 80.9%


## 3 · SVM classification (5-fold cross-validation)

In [4]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=False, random_state=42)
y_pred_svm = cross_val_predict(svm, X_pca, y, cv=cv)

micro_f1_svm = f1_score(y, y_pred_svm, average="micro")
macro_f1_svm = f1_score(y, y_pred_svm, average="macro")
print(f"SVM  micro-F1 : {micro_f1_svm:.4f}")
print(f"SVM  macro-F1 : {macro_f1_svm:.4f}")

SVM  micro-F1 : 0.6198
SVM  macro-F1 : 0.5914


## 4 · Confusion matrix

In [5]:
from sklearn.metrics import confusion_matrix

cm_mat = confusion_matrix(y, y_pred_svm)
n_classes = cm_mat.shape[0]
class_names = [f"C{c:02d}" for c in range(n_classes)]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_mat, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45, fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"SVM Confusion Matrix  (micro-F1 = {micro_f1_svm:.3f})")
# Annotate cells
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(
            j, i, str(cm_mat[i, j]),
            ha="center", va="center", fontsize=7,
            color="white" if cm_mat[i, j] > cm_mat.max() * 0.5 else "black",
        )
plt.tight_layout()

out_path = Path("outputs/notebook_02_confusion_matrix.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {out_path}")

Saved → outputs\notebook_02_confusion_matrix.png

## 5 · CNN classification (15 epochs demonstration)

In [6]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as f1
from src.models.deep_learning import Conv1DClassifier, train_classifier, _get_safe_device

device = _get_safe_device()
X_tr, X_val, y_tr, y_val = train_test_split(
    X_pca, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_tr.shape}  Val: {X_val.shape}")

cnn = Conv1DClassifier(n_classes=n_classes, n_filters=32, kernel_size=3, dropout=0.2, lr=1e-3)
trained_cnn = train_classifier(
    cnn, X_tr, y_tr, X_val, y_val, epochs=15, batch_size=32, device=device
)

# Evaluate on validation set
trained_cnn.eval()
with torch.no_grad():
    X_val_t = torch.FloatTensor(X_val).unsqueeze(1).to(device)
    logits = trained_cnn(X_val_t).cpu().numpy()
y_pred_cnn = logits.argmax(axis=1)

cnn_f1 = f1(y_val, y_pred_cnn, average="micro")
print(f"CNN val micro-F1 (15 epochs) : {cnn_f1:.4f}")

Train: (153, 50)  Val: (39, 50)


CNN val micro-F1 (15 epochs) : 0.3846


## 6 · SHAP wavelength importance

We use **LinearSVC** (fast, exact SHAP) trained on PCA(50) features,
then project the 50-dimensional SHAP vector back to the 40 002-channel wavelength axis
via the PCA loading matrix.

In [7]:
import shap
from scipy.ndimage import uniform_filter1d
from sklearn.svm import LinearSVC

# Train LinearSVC on full dataset (for SHAP analysis only)
lsvc = LinearSVC(C=1.0, max_iter=3000, random_state=42)
lsvc.fit(X_pca, y)

# SHAP values in PCA space — explains 30 samples
explainer = shap.LinearExplainer(lsvc, X_pca, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_pca[:30])   # (30, 50, n_classes)

# Aggregate: mean |SHAP| over samples and classes → (50,) feature importance
shap_arr = np.array(shap_values)               # (30, 50, n_classes)
shap_pca = np.abs(shap_arr).mean(axis=0).mean(axis=1)   # (50,)

# Project PCA SHAP back to wavelength space: (40002, 50) @ (50,) = (40002,)
shap_wl = np.abs(pca.components_.T @ shap_pca)   # (40002,)

# Smooth over ~200 channels (~4 nm) for readability
shap_smooth = uniform_filter1d(shap_wl, size=200)
shap_norm   = shap_smooth / shap_smooth.max()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(wl, shap_norm, alpha=0.35, color="darkorange", label="SHAP importance")
ax.plot(wl, shap_norm, color="darkorange", lw=0.9)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Relative SHAP importance")
ax.set_title("SHAP Feature Importance — Projected to Wavelength Space (LinearSVC on PCA features)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

out_path = Path("outputs/notebook_02_shap_overlay.png")
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {out_path}")

Saved → outputs\notebook_02_shap_overlay.png


## Summary

| Model | micro-F1 (5-fold CV) |
|-------|----------------------|
| SVM (RBF, C=10) | See above |
| CNN (15 epochs, demonstration) | See above (val) |

- SHAP analysis identifies which spectral regions drive class separation.
- Both SVM and CNN operate on PCA(50) features of SNV-normalised spectra.